In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import mlflow
import mlflow.catboost
from catboost import CatBoostClassifier
import psycopg
from autofeat import AutoFeatClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss


TABLE_NAME = 'clean_users_churn' # таблица с данными

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

In [20]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

In [21]:
df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2133,3023-GFLBR,2017-03-01,2019-12-01,Month-to-month,No,Credit card (automatic),86.15,2745.7,Fiber optic,...,No,No,No,Yes,Female,0,Yes,Yes,Yes,1
1,837,0727-BMPLR,2015-04-01,2019-11-01,One year,Yes,Electronic check,100.00,5509.3,Fiber optic,...,Yes,No,Yes,Yes,Female,1,No,No,Yes,1


In [42]:
exclude_columns = ['id', 'customer_id', 'target', 'begin_date', 'end_date']
features_no_target = [col for col in df.columns if col not in exclude_columns]
target = ['target']

split_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features_no_target],
    df[target],
    test_size=test_size,
    shuffle=False,
    random_state=42
)

cat_features = [
    'paperless_billing',
    'payment_method',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'gender',
    'senior_citizen',
    'partner',
    'dependents',
    'multiple_lines',
    'type'
]
num_features = ["monthly_charges", "total_charges"]

features = cat_features + num_features

transformations = ('1/', 'log', 'abs', 'sqrt')

afc = AutoFeatClassifier(categorical_cols=cat_features,
                         transformations=transformations,
                         feateng_steps=1,
                         n_jobs=-1,
                         verbose=1,)

X_train_features = afc.fit_transform(X_train, y_train)
X_test_features = afc.transform(X_test)

/home/mle-user/mle-mlflow/.venv_mle_mlflow/lib/python3.10/site-packages/sklearn/utils/validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
2024-11-08 15:57:50,380 INFO: [AutoFeat] The 1 step feature engineering process could generate up to 140 features.
2024-11-08 15:57:50,380 INFO: [AutoFeat] With 5615 data points this new feature matrix would use about 0.00 gb of space.
2024-11-08 15:57:50,384 INFO: [feateng] Step 1: transformation of original features


2024-11-08 15:57:51,330 INFO: [feateng] Generated 6 transformed features from 35 original features - done.
2024-11-08 15:57:51,335 INFO: [feateng] Generated altogether 6 new features in 1 steps
2024-11-08 15:57:51,336 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2024-11-08 15:57:51,388 INFO: [feateng] Generated a total of 3 additional features


[featsel] Scaling data...done.
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.


2024-11-08 15:57:54,349 INFO: [featsel] Feature selection run 2/5
2024-11-08 15:57:54,357 INFO: [featsel] Feature selection run 1/5


[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   12.4s


...

[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   21.4s remaining:   14.3s


2024-11-08 15:58:12,803 INFO: [featsel] Feature selection run 5/5
2024-11-08 15:58:18,014 INFO: [featsel] 6 features after 5 feature selection runs
2024-11-08 15:58:18,018 INFO: [featsel] 6 features after correlation filtering


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   26.6s finished


2024-11-08 15:58:18,643 INFO: [featsel] 6 features after noise filtering
2024-11-08 15:58:18,644 INFO: [AutoFeat] Computing 1 new features.
2024-11-08 15:58:18,783 INFO: [AutoFeat]     1/    1 new features ...done.
2024-11-08 15:58:18,787 INFO: [AutoFeat] Final dataframe with 36 feature columns (19 new).
2024-11-08 15:58:18,787 INFO: [AutoFeat] Training final classification model.


2024-11-08 15:58:19,203 INFO: [AutoFeat] Trained model: largest coefficients:
2024-11-08 15:58:19,204 INFO: [-2.48430417]
2024-11-08 15:58:19,205 INFO: 1.023660 * cat_type_Month-to-month
2024-11-08 15:58:19,206 INFO: 0.897509 * cat_type_Two year
2024-11-08 15:58:19,207 INFO: 0.603023 * cat_payment_method_Electronic check
2024-11-08 15:58:19,207 INFO: 0.035045 * monthly_charges
2024-11-08 15:58:19,208 INFO: 0.003093 * 1/total_charges
2024-11-08 15:58:19,209 INFO: 0.000298 * total_charges
2024-11-08 15:58:19,215 INFO: [AutoFeat] Final score: 0.7325
2024-11-08 15:58:19,256 INFO: [AutoFeat] Computing 1 new features.
2024-11-08 15:58:19,259 INFO: [AutoFeat]     1/    1 new features ...done.


In [44]:
X_train_features.columns

Index(['monthly_charges', 'total_charges', 'cat_paperless_billing_No',
       'cat_paperless_billing_Yes',
       'cat_payment_method_Bank transfer (automatic)',
       'cat_payment_method_Credit card (automatic)',
       'cat_payment_method_Electronic check',
       'cat_payment_method_Mailed check', 'cat_internet_service_DSL',
       'cat_internet_service_Fiber optic', 'cat_online_security_No',
       'cat_online_security_Yes', 'cat_online_backup_No',
       'cat_online_backup_Yes', 'cat_device_protection_No',
       'cat_device_protection_Yes', 'cat_tech_support_No',
       'cat_tech_support_Yes', 'cat_streaming_tv_No', 'cat_streaming_tv_Yes',
       'cat_streaming_movies_No', 'cat_streaming_movies_Yes',
       'cat_gender_Female', 'cat_gender_Male', 'cat_senior_citizen_0',
       'cat_senior_citizen_1', 'cat_partner_No', 'cat_partner_Yes',
       'cat_dependents_No', 'cat_dependents_Yes', 'cat_multiple_lines_No',
       'cat_multiple_lines_Yes', 'cat_type_Month-to-month',
       'c